# AgroVision AI – Train the plant disease model on Google Colab (free GPU)

1. `Runtime → Change runtime type → GPU`
2. Upload the `ml_training/` folder (or the whole project zip) to Colab
3. Run the cells in order. Training takes ~30–60 min on a T4 GPU.


In [ ]:
!nvidia-smi -L
!pip -q install kaggle

## 1. Get the project files
Upload `agrovision-ai.zip` with the file browser on the left, then unzip:

In [ ]:
!unzip -q -o agrovision-ai.zip -d /content && ls /content/agrovision-ai
%cd /content/agrovision-ai/ml_training

## 2. Download PlantVillage from Kaggle
Create an API token at kaggle.com → Settings → *Create New Token* and upload `kaggle.json`.

In [ ]:
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p dataset --unzip -q
!ls dataset

`train.py` searches `dataset/` recursively and uses only the **color** images of the 27 supported classes (6 crops). If both `color/` and `grayscale/` exist, point `--data-dir` at the color folder.

In [ ]:
!python train.py --data-dir "dataset/plantvillage dataset/color" --arch mobilenetv2 --epochs 10 --fine-tune-epochs 10 --batch-size 64

In [ ]:
!python evaluate.py
from IPython.display import Image; Image('artifacts/confusion_matrix.png')

In [ ]:
!python export_model.py --version 1.0.0
!cd ../backend/trained_models && zip -q -r /content/agrovision_trained_model.zip . && ls -la

## 3. Download
Download `/content/agrovision_trained_model.zip` and extract it into `backend/trained_models/` on your machine. Restart the backend – `GET /api/detection/model-info` should now show `ready: true`.

In [ ]:
from google.colab import files; files.download('/content/agrovision_trained_model.zip')